## Import Needed Libraries

In [ ]:
import model
import data
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import time
import matplotlib.pyplot as plt

## Load Dataset

In [ ]:
train_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_train.npy", ks_npy="../Data/ks_train.npy")
val_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_val.npy", ks_npy="../Data/ks_val.npy")
test_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_test.npy", ks_npy="../Data/ks_test.npy")


In [ ]:
# Check loaded file sizes

print('Training x Input Arrays Size:', train_dataset.shape)
print('Training kx Input Array Size:', val_dataset.shape)
print('Training Output Array Size:', test_dataset.shape)


## Dataset Preparation

In [ ]:
class Custom_Dataset(Dataset):
    def __init__(self, data, labels):

        self.data = torch.Tensor(data).to(torch.float32).to("cuda")
        self.labels = torch.Tensor(labels).to(torch.float32).to("cuda")

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):
        # Return the data as a pytorch tensor
        return self.data[idx].to("cuda"), self.labels[idx].to("cuda")


## Define Hyper/parameters

In [ ]:
size_kernel = [3, 5, 10, 15] # kernel size, set params_user to 0
size_batch = [16, 32, 64, 100] # batch size, set params_user to 1
reg_param = [1e-2, 1e-3, 1e-4] # regularization parameter, set params_user to 2

params_user = 1 # choose between 0, 1, 2, 3

N_train = 5400
N_test = 1001
N_epochs = 500 # can vary

# Define loss criteria

criterion = nn.MSELoss()

# total runtime

time_total = time.time()

# reshape inputs into 3D arrays

train_size = [N_train, len(train_omega), len(train_kx)]
test_size = [N_test, len(train_omega), len(train_kx)]
train_inputs = np.expand_dims(np.reshape(train_x, train_size), axis=1)
test_inputs = np.expand_dims(np.reshape(test_x, test_size), axis=1)

# scale labels

train_labels = train_labels * (train_kx[-1] - train_kx[0]) / (train_omega[-1] - train_omega[0])
test_labels = test_labels * (test_kx[-1] - test_kx[0]) / (test_omega[-1] - test_omega[0])

# clear memory
del train_x
del test_x

print(train_inputs.shape,test_inputs.shape)

# preparing the training data

train_dataset = Custom_Dataset(data = train_inputs, labels = train_labels)
train_loader = DataLoader(train_dataset, batch_size = N_batch, shuffle = True)

print(train_dataset.shape, train_loader.shape)

## Model Instantiation

In [ ]:
model = CNN(N_channel = 1, kernel_s = k)

## Training Loop

In [ ]:
# Parameter study case: select which parameter to vary

if params_user == 0:
    params = size_kernel
elif params_user == 1:
    params = size_batch
elif params_user == 2:
    params = reg_param

train_loss_metrics = []
test_loss_metrics = []
train_accuracy_metrics = []
test_accuracy_metrics = []

param_time = []

for param in params:

    if params_user == 0:
        k = param
        N_batch = 100
        reg_param = 1e-3
    elif params_user == 1:
        N_batch = param
        k = 3
        reg_param = 1e-3
    elif params_user == 2:
        reg_param = param
        k = 3
        N_batch = 100

    # Per epoch
    
    train_loss_history = []
    test_loss_history = []
    train_accuracy_history = []
    test_accuracy_history = []

    # Calculate and display the number of mini-batches

    num_minibatches = len(train_loader)

    print(f"Number of Mini-Batches: {num_minibatches}")

    t_0 = time.time()

    t_0_main = time.time()

    # initialize model

    model = Net(N_channel = 1, kernel_s = k).to("cuda")

    # Define optimizer. Use ADAM with regularization parameter as weight decay
    optimizer = optim.Adam(model.parameters(), weight_decay = reg_param)

    for epoch in range(1, N_epochs + 1):
    #for epoch in range(2):
        for batch in train_loader:
            x_batch = batch[0]
            y_batch = batch[1]

            # zero the gradients
            
            optimizer.zero_grad()
            
            # forward
            
            y_pred = model(x_batch).to("cuda")

            # compute loss
            
            loss = criterion(y_pred[:,0], y_batch[:,0]) + lambda_weight * criterion(y_pred[:,1], y_batch[:,1])
            
            # back propagation
            
            loss.backward()
            
            # update the parameters theta
            
            optimizer.step()

            #acc = torch.abs(y_pred - y_batch)/y_batch
            #acc[~torch.isfinite(acc)] = 0
            #accuracy = torch.nanmean(acc)

            err = y_pred - y_batch
            error = err.cpu().detach().numpy()
            y_b = y_batch.cpu().detach().numpy()
            accuracy = np.linalg.norm(error) / np.linalg.norm(y_b)
            
            # removed .to("cpu").detach().numpy() for now
            train_accuracy_history.append(accuracy)
            train_loss_history.append(loss.item())

            with torch.no_grad():
                
                model.eval()

                y_test_pred = model(torch.Tensor(test_inputs).to(torch.float32).to("cuda"))
                
                # loss is calculated using viscous units
                
                test_loss_mean = criterion(y_test_pred[:,0]/utau_test, torch.Tensor(test_labels[:,0]/utau_train).to(torch.float32).to('cuda'))
                test_loss_rms = criterion(y_test_pred[:,1]/utau_test, torch.Tensor(test_labels[:,1]/utau_train).to(torch.float32).to('cuda'))
                test_loss = test_loss_mean + lambda_weight * test_loss_rms
                test_loss_history.append(test_loss.item())

                test_acc = np.abs(y_test_pred.to("cpu").detach().numpy() - test_labels)/test_labels
                test_acc[~np.isfinite(test_acc)] = 0
                test_accuracy = np.nanmean(test_acc)
                test_accuracy_history.append(test_accuracy)

            model.train()

        if epoch % int(N_epochs/5) == 0: # asumming N_epochs/20 is an integer
            print(f"Epoch [{epoch}/{int(N_epochs)}],")
            print(f"Data: Loss: {loss.item()} Accuracy: {accuracy}")

    train_loss_metrics.append(np.mean(train_loss_history[-1000:]))
    test_loss_metrics.append(np.mean(test_loss_history[-1000:]))
    train_accuracy_metrics.append(np.mean(train_accuracy_history[-1000:]))
    test_accuracy_metrics.append(np.mean(test_accuracy_history[-1000:]))
    
    plt.figure(figsize = (8,6))
    plt.box(True)
    plt.plot(train_loss_history, label = "Training Loss")
    plt.plot(test_loss_history, label = "Test Loss")
    plt.gca().set_yscale('log')
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Loss vs. Epoch for Lambda = {param}')
    plt.show()

    final_train_loss = train_loss_history[-1]
    final_test_loss = test_loss_history[-1]

    print(f"Final Losses for Lambda = {param}: Training Loss - {final_train_loss}, Test Loss - {final_test_loss}")
    
    t_1_main = time.time()

    print(f"Runtime {t_1_main-t_0_main} s.")
    param_time.append(t_1_main-t_0_main)
    